In [ ]:

import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CSV_PATH = "dataset_clean_v2.csv"
TARGET = "orders_total"
DATE_COLUMN = "date"
TEST_DAYS = 20
RIDGE_ALPHA = 10.0
FUTURE_DAYS = 14
NON_ORDER_ASSUMPTION = "rolling_mean_7"   # "rolling_mean_7" or "last_value"

# ============================================================
# SECTION 1: LOAD AND PREPARE DATASET
# ============================================================

df = pd.read_csv(CSV_PATH)
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
df = df.sort_values(DATE_COLUMN).reset_index(drop=True)

for lag in [1, 2, 3, 7, 14]:
    df[f"orders_lag_{lag}"] = df[TARGET].shift(lag)

df["orders_roll_7"] = df[TARGET].shift(1).rolling(window=7).mean()
df["orders_roll_14"] = df[TARGET].shift(1).rolling(window=14).mean()

df["day_of_week"] = df[DATE_COLUMN].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 4).astype(int)

weekday_dummies = pd.get_dummies(df["day_of_week"], prefix="weekDay", dtype=float)
for i in range(7):
    column = f"weekDay_{i}"
    if column not in weekday_dummies.columns:
        weekday_dummies[column] = 0.0
weekday_dummies = weekday_dummies[[f"weekDay_{i}" for i in range(7)]]
df = pd.concat([df, weekday_dummies], axis=1)

features_to_delete = [
    "orders_data_available", "orders_completed", "orders_canceled", "orders_pending",
    "unique_order_users", "cancelled_orders", "successful_orders", "kabul_order_count",
    "herat_order_count", "online_payment_orders", "average_order_item_count",
    "average_order_value", "first_orders_count", "repeat_orders_count",
    "meta_leads", "meta_cpc_usd", "meta_cpm_usd", "meta_active_campaigns",
    "meta_results_reported", "google_ads_cost_usd_estimated", "google_ads_avg_cpc_usd",
    "blogger_spend_afn", "total_ad_spend_afn",
    "delivery_share_lag7_avg", "delivery_share_lag1", "avg_delivery_fee_lag7_avg",
    "avg_delivery_fee_lag1", "avg_discount_amount_lag1", "avg_discount_amount_lag7_avg",
    "new_active_restaurants", "rating_sum_that_day", "featured_that_day",
    "commission_sum_that_day", "new_available_dishes", "count_that_day",
    "new_featured_dishes", "cumulative_featured_restaurants", "cumulative_commission_sum",
    "avg_commission_rate",
    "blogger_views_reach_lag7_avg", "blogger_views_reach_lag1",
    "spend_lag_1", "meta_impressions_lag1", "meta_impressions_lag7_avg",
    "meta_link_clicks_lag1", "meta_ctr_percent_lag7_avg", "meta_cpm_usd_lag1",
    "meta_cpc_usd_lag7_avg",
    "google_ads_ctr_percent_estimated", "google_ads_ctr_percent_estimated_lag1",
    "google_ads_ctr_percent_estimated_lag7_avg",
    "avg_item_price_lag1", "avg_item_price_lag7_avg",
    "avg_order_food_value_lag1", "avg_order_food_value_lag7_avg",
    "day_of_week_num",
    "cumulative_users", "cumulative_active_restaurants", "cumulative_restaurant_count",
    "cumulative_rating_sum", "cumulative_available_dishes", "cumulative_featured_dishes",
]
features_to_delete = [c for c in set(features_to_delete) if c in df.columns]
df = df.drop(columns=features_to_delete, errors="ignore")

numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
for col in [TARGET, "day_of_week", "Unnamed: 0"]:
    if col in numeric_columns:
        numeric_columns.remove(col)
FEATURES = numeric_columns

ORDER_DERIVED_FEATURES = [f"orders_lag_{lag}" for lag in [1, 2, 3, 7, 14]] + ["orders_roll_7", "orders_roll_14"]
CALENDAR_FEATURES = ["is_weekend"] + [f"weekDay_{i}" for i in range(7)]
ASSUMPTION_FEATURES = [f for f in FEATURES if f not in ORDER_DERIVED_FEATURES and f not in CALENDAR_FEATURES]

print(df.head())
print(f"\nFeatures ({len(FEATURES)}): {FEATURES}")

# ============================================================
# SECTION 2: TRAINING ON HISTORICAL DATA THAT HAS THE TARGET
# ============================================================

test_start = len(df) - TEST_DAYS
train = df.iloc[:test_start].copy()
test = df.iloc[test_start:].copy()

train_model = train.dropna(subset=[f"orders_lag_{lag}" for lag in [1, 2, 3, 7, 14]]).copy()
X_train = train_model[FEATURES]
y_train = train_model[TARGET]

eval_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=RIDGE_ALPHA)),
])
eval_model.fit(X_train, y_train)

history_eval = {i: df.loc[i, TARGET] for i in range(test_start)}

def build_row_from_existing(row_index, history):
    current_row = df.loc[row_index]
    values = {f: current_row[f] for f in FEATURES if f not in ORDER_DERIVED_FEATURES}
    for lag in [1, 2, 3, 7, 14]:
        values[f"orders_lag_{lag}"] = history.get(row_index - lag, np.nan)
    prev7 = [v for v in (history.get(row_index - k, np.nan) for k in range(1, 8)) if pd.notna(v)]
    values["orders_roll_7"] = np.mean(prev7) if prev7 else np.nan
    prev14 = [v for v in (history.get(row_index - k, np.nan) for k in range(1, 15)) if pd.notna(v)]
    values["orders_roll_14"] = np.mean(prev14) if prev14 else np.nan
    return pd.DataFrame([values], columns=FEATURES)

test_predictions = []
for i in range(test_start, len(df)):
    X_row = build_row_from_existing(i, history_eval)
    pred = max(0, eval_model.predict(X_row)[0])
    test_predictions.append(pred)
    history_eval[i] = pred

results = test[[DATE_COLUMN, TARGET]].copy()
results["predicted"] = test_predictions
results["abs_error"] = (results["predicted"] - results[TARGET]).abs()

mae = mean_absolute_error(results[TARGET], results["predicted"])
rmse = np.sqrt(mean_squared_error(results[TARGET], results["predicted"]))
r2 = r2_score(results[TARGET], results["predicted"])
non_zero = results[TARGET] != 0
mape = (results.loc[non_zero, "abs_error"] / results.loc[non_zero, TARGET].abs()).mean() * 100
acc = np.clip(100 - mape, 0, 100)

print("\n--- Accuracy on held-out historical days ---")
print(results.to_string(index=False))
print(f"Accuracy: {acc:.2f}%  |  MAE: {mae:.2f}  |  RMSE: {rmse:.2f}  |  R2: {r2:.4f}")

# ============================================================
# SECTION 3: PREDICTING FUTURE DATA (BEYOND THE DATASET)
# ============================================================

full_model_data = df.dropna(subset=[f"orders_lag_{lag}" for lag in [1, 2, 3, 7, 14]]).copy()
X_full = full_model_data[FEATURES]
y_full = full_model_data[TARGET]

final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=RIDGE_ALPHA)),
])
final_model.fit(X_full, y_full)

if NON_ORDER_ASSUMPTION == "rolling_mean_7":
    assumption_values = df[ASSUMPTION_FEATURES].tail(7).mean()
else:
    assumption_values = df[ASSUMPTION_FEATURES].iloc[-1]

last_date = df[DATE_COLUMN].max()
future_dates = [last_date + pd.Timedelta(days=i) for i in range(1, FUTURE_DAYS + 1)]

history = {i: df.loc[i, TARGET] for i in range(len(df))}
start_index = len(df)

def build_future_row(row_index, date, history):
    values = {}
    dow = date.dayofweek
    values["is_weekend"] = int(dow >= 4)
    for i in range(7):
        values[f"weekDay_{i}"] = 1.0 if dow == i else 0.0
    for lag in [1, 2, 3, 7, 14]:
        values[f"orders_lag_{lag}"] = history.get(row_index - lag, np.nan)
    prev7 = [v for v in (history.get(row_index - k, np.nan) for k in range(1, 8)) if pd.notna(v)]
    values["orders_roll_7"] = np.mean(prev7) if prev7 else np.nan
    prev14 = [v for v in (history.get(row_index - k, np.nan) for k in range(1, 15)) if pd.notna(v)]
    values["orders_roll_14"] = np.mean(prev14) if prev14 else np.nan
    for feature in ASSUMPTION_FEATURES:
        values[feature] = assumption_values[feature]
    return pd.DataFrame([values], columns=FEATURES)

future_predictions = []
for offset, date in enumerate(future_dates):
    row_index = start_index + offset
    X_future = build_future_row(row_index, date, history)
    pred = max(0, final_model.predict(X_future)[0])
    future_predictions.append(pred)
    history[row_index] = pred

future_results = pd.DataFrame({
    DATE_COLUMN: future_dates,
    "predicted_orders_total": np.round(future_predictions, 1),
})

print("\n--- Future forecast ---")
print(future_results.to_string(index=False))

future_results.to_csv("future_orders_forecast.csv", index=False)